# Búsqueda de días con transición convectiva en Bogotá (SKBO)

Este notebook descarga reportes METAR del aeropuerto El Dorado (SKBO) desde OGIMET
y busca días donde se observa la secuencia:

**Cielo despejado → Desarrollo de nube convectiva (TCU) → Tormenta/lluvia fuerte (CB, TSRA)**

Esa secuencia representa el ciclo diurno típico de convección en la Sabana de Bogotá,
que es exactamente el fenómeno que la tesis busca predecir con anticipación de 4-6 horas.

## Celda 1 — Librerías

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import re
import time

## Celda 2 — Parámetros de búsqueda
Aquí se definen qué meses se van a consultar.

Cambiar `MESES_A_CONSULTAR` para ampliar o reducir la búsqueda.

In [2]:
# Estación meteorológica: El Dorado, Bogotá
ESTACION = 'SKBO'

# Meses a consultar: lista de tuplas (año, mes)
# Bogotá tiene dos temporadas de lluvia: marzo-mayo y septiembre-noviembre
# Empezamos con los meses más lluviosos de 2024
MESES_A_CONSULTAR = [
    (2024, 3),   # marzo 2024
    (2024, 4),   # abril 2024
    (2024, 10),  # octubre 2024
]

# Palabras clave para cada etapa de la secuencia convectiva
CLAVES_DESPEJADO  = ['SKC', 'FEW', 'NSC', 'CAVOK']          # cielo limpio o pocas nubes
CLAVES_DESARROLLO = ['TCU']                                 # cumulonimbo en desarrollo
CLAVES_TORMENTA   = ['CB', 'TSRA', '+TSRA', '+RA', 'TS']    # tormenta activa

print('Parámetros cargados correctamente.')
print(f'Estación: {ESTACION}')
print(f'Meses a consultar: {MESES_A_CONSULTAR}')

Parámetros cargados correctamente.
Estación: SKBO
Meses a consultar: [(2024, 3), (2024, 4), (2024, 10)]


## Celda 3 — Función de descarga
Esta función le pide a OGIMET los reportes METAR de un mes completo
y los devuelve como una lista de líneas de texto.

In [6]:
def descargar_metars(estacion, año, mes):
    """
    Descarga reportes METAR de OGIMET para una estación y mes dado.
    Usa el endpoint correcto: ogimet.com/cgi-bin/getmetar
    Devuelve una lista de strings, uno por reporte.
    """
    from datetime import datetime
    import calendar

    # Primer y último momento del mes
    ultimo_dia = calendar.monthrange(año, mes)[1]
    begin = f"{año}{mes:02d}010000"
    end   = f"{año}{mes:02d}{ultimo_dia:02d}2359"

    # URL correcta según documentación de OGIMET
    url = f"http://www.ogimet.com/cgi-bin/getmetar?icao={estacion}&begin={begin}&end={end}&lang=eng"

    print(f'Descargando {estacion} {año}-{mes:02d}...')
    print(f'  URL: {url}')

    try:
        respuesta = requests.get(url, timeout=30)
    except Exception as e:
        print(f'  Error de conexión: {e}')
        return []

    if respuesta.status_code != 200:
        print(f'  Error HTTP {respuesta.status_code}')
        return []

    # El formato devuelto es CSV: ICAOIND,YEAR,MONTH,DAY,HOUR,MIN,REPORT
    # Extraemos solo la columna REPORT (última columna)
    metars = []
    for linea in respuesta.text.split('\n'):
        linea = linea.strip()
        if linea and not linea.startswith('#'):
            partes = linea.split(',')
            if len(partes) >= 7:
                reporte = partes[6].strip()
                if reporte.startswith('METAR') or reporte.startswith(estacion):
                    metars.append(reporte)

    print(f'  {len(metars)} reportes encontrados.')
    return metars

print('Función de descarga corregida y lista.')

Función de descarga corregida y lista.


## Celda 4 — Función de análisis de un día
Esta función toma todos los reportes de un día y decide si ese día
tuvo la secuencia convectiva que se busca.

In [7]:
def clasificar_reporte(metar):
    """
    Clasifica un reporte METAR en una de tres categorías:
    - 'despejado'  : cielo limpio o pocas nubes, sin convección
    - 'desarrollo' : TCU presente (cumulonimbo en desarrollo)
    - 'tormenta'   : CB, TSRA o lluvia fuerte presente
    - 'otro'       : no encaja en ninguna categoría relevante
    """
    for clave in CLAVES_TORMENTA:
        if clave in metar:
            return 'tormenta'
    for clave in CLAVES_DESARROLLO:
        if clave in metar:
            return 'desarrollo'
    for clave in CLAVES_DESPEJADO:
        if clave in metar:
            return 'despejado'
    return 'otro'


def analizar_dia(reportes_del_dia):
    """
    Recibe una lista de reportes METAR de un mismo día.
    Devuelve True si ese día tuvo la secuencia: despejado → tormenta.
    También acepta: despejado → desarrollo → tormenta.
    """
    categorias = [clasificar_reporte(r) for r in reportes_del_dia]

    tuvo_despejado = 'despejado' in categorias
    tuvo_tormenta  = 'tormenta'  in categorias

    if not (tuvo_despejado and tuvo_tormenta):
        return False, categorias

    # Verificar que el despejado ocurrió ANTES de la tormenta
    primer_despejado = categorias.index('despejado')
    ultimo_tormenta  = len(categorias) - 1 - categorias[::-1].index('tormenta')

    secuencia_valida = primer_despejado < ultimo_tormenta
    return secuencia_valida, categorias


print('Funciones de análisis listas.')

Funciones de análisis listas.


## Celda 5 — Búsqueda principal
Aquí se descarga todo y se identifican los días candidatos.

In [8]:
dias_candidatos = []  # aquí guardamos los días que cumplen la secuencia
todos_los_metars = [] # aquí guardamos todos los reportes para análisis posterior

for (año, mes) in MESES_A_CONSULTAR:

    metars = descargar_metars(ESTACION, año, mes)
    todos_los_metars.extend(metars)

    # Agrupar reportes por día
    # El día está en el timestamp: METAR SKBO DDHHMMZ ...
    # Los primeros dos dígitos del timestamp son el día del mes
    reportes_por_dia = {}
    for metar in metars:
        # Extraer el timestamp (ej: '072300Z' -> día '07')
        match = re.search(r'SKBO (\d{2})(\d{2})(\d{2})Z', metar)
        if match:
            dia = int(match.group(1))
            fecha = datetime(año, mes, dia)
            if fecha not in reportes_por_dia:
                reportes_por_dia[fecha] = []
            reportes_por_dia[fecha].append(metar)

    # Analizar cada día
    for fecha, reportes in sorted(reportes_por_dia.items()):
        es_candidato, categorias = analizar_dia(reportes)
        if es_candidato:
            n_reportes   = len(reportes)
            n_despejado  = categorias.count('despejado')
            n_desarrollo = categorias.count('desarrollo')
            n_tormenta   = categorias.count('tormenta')
            dias_candidatos.append({
                'fecha':       fecha.strftime('%Y-%m-%d'),
                'reportes':    n_reportes,
                'despejado':   n_despejado,
                'desarrollo':  n_desarrollo,
                'tormenta':    n_tormenta,
                'secuencia':   ' → '.join([c for c in categorias if c != 'otro'])
            })

    time.sleep(2)  # pausa entre consultas para no saturar el servidor de OGIMET

print(f'\nBúsqueda completada.')
print(f'Total de días candidatos encontrados: {len(dias_candidatos)}')

Descargando SKBO 2024-03...
  URL: http://www.ogimet.com/cgi-bin/getmetar?icao=SKBO&begin=202403010000&end=202403312359&lang=eng
  744 reportes encontrados.
Descargando SKBO 2024-04...
  URL: http://www.ogimet.com/cgi-bin/getmetar?icao=SKBO&begin=202404010000&end=202404302359&lang=eng
  721 reportes encontrados.
Descargando SKBO 2024-10...
  URL: http://www.ogimet.com/cgi-bin/getmetar?icao=SKBO&begin=202410010000&end=202410312359&lang=eng
  Error HTTP 501

Búsqueda completada.
Total de días candidatos encontrados: 22


## Celda 6 — Resultados
Tabla con los días candidatos ordenados por intensidad de la secuencia
(más horas de tormenta = caso más claro).

In [9]:
if len(dias_candidatos) == 0:
    print('No se encontraron días candidatos en los meses consultados.')
    print('Prueba ampliar MESES_A_CONSULTAR en la Celda 2.')
else:
    df = pd.DataFrame(dias_candidatos)
    df = df.sort_values('tormenta', ascending=False)

    print('Días con secuencia convectiva (despejado → tormenta):')
    print('=' * 70)
    print(df.to_string(index=False))
    print('=' * 70)
    print()
    print('Los primeros de la lista son los mejores candidatos:')
    print('más horas de tormenta = evento convectivo más claro y prolongado.')

Días con secuencia convectiva (despejado → tormenta):
     fecha  reportes  despejado  desarrollo  tormenta                                                                                                                                                                                                             secuencia
2024-03-21        24          2           4         7                                                                tormenta → tormenta → tormenta → desarrollo → despejado → despejado → desarrollo → desarrollo → desarrollo → tormenta → tormenta → tormenta → tormenta
2024-04-01        24          2           1         7                                                                                                       despejado → despejado → desarrollo → tormenta → tormenta → tormenta → tormenta → tormenta → tormenta → tormenta
2024-04-18        24         13           0         5      despejado → despejado → despejado → despejado → despejado → despejado → despejado →

## Celda 7 — Ver el detalle de un día candidato
Cambia la fecha abajo para ver hora por hora qué pasó ese día.

In [10]:
# Cambia esta fecha por cualquiera de los candidatos de la tabla anterior
FECHA_A_REVISAR = '2024-04-18'  # ejemplo: ajusta según los resultados

año_r = int(FECHA_A_REVISAR[:4])
mes_r = int(FECHA_A_REVISAR[5:7])
dia_r = int(FECHA_A_REVISAR[8:10])

print(f'Reportes del {FECHA_A_REVISAR} (hora UTC):')
print('=' * 80)

for metar in todos_los_metars:
    match = re.search(r'SKBO (\d{2})(\d{2})(\d{2})Z', metar)
    if match:
        dia_m = int(match.group(1))
        hora  = match.group(2)
        if dia_m == dia_r:
            # Verificar que es del mes y año correctos
            categoria = clasificar_reporte(metar)
            etiqueta = {'despejado': '☀️ ', 'desarrollo': '⛅ TCU', 'tormenta': '⛈️  TORMENTA', 'otro': '🌥️ '}
            print(f"{hora}:00 UTC  {etiqueta[categoria]:15}  {metar[:80]}")

print('=' * 80)
print('Hora Bogotá = Hora UTC - 5h')

Reportes del 2024-04-18 (hora UTC):
00:00 UTC  ☀️               METAR SKBO 180000Z 13007KT 9999 FEW030 17/09 Q1025 NOSIG=
01:00 UTC  ☀️               METAR SKBO 180100Z 09004KT 9999 FEW017 15/10 Q1026 NOSIG=
02:00 UTC  ☀️               METAR SKBO 180200Z 10005KT 9999 FEW017 15/09 Q1027 NOSIG=
03:00 UTC  ☀️               METAR SKBO 180300Z 09004KT 9999 FEW020 14/09 Q1027 NOSIG=
04:00 UTC  ☀️               METAR SKBO 180400Z 03005KT 9999 FEW020 12/10 Q1027 NOSIG=
05:00 UTC  🌥️               METAR SKBO 180500Z 04004KT 9999 BKN025 13/10 Q1027 NOSIG=
06:00 UTC  🌥️               METAR SKBO 180600Z 03005KT 9999 BKN025 12/10 Q1027 NOSIG=
07:00 UTC  🌥️               METAR SKBO 180700Z 07004KT 9999 BKN025 12/10 Q1027 NOSIG=
08:00 UTC  🌥️               METAR SKBO 180800Z 05003KT 9999 SCT025 11/09 Q1026 NOSIG=
09:00 UTC  🌥️               METAR SKBO 180900Z VRB02KT 9999 SCT025 11/09 Q1026 NOSIG=
10:00 UTC  🌥️               METAR SKBO 181000Z 05004KT 9999 BKN025 13/10 Q1027 NOSIG=
11:00 UTC  🌥️     

## Celda 8 — Buscar un día completamente despejado (control)
Para el día de contraste (sin lluvia), buscamos días sin ninguna señal convectiva.

In [ ]:
dias_despejados = []

for (año, mes) in MESES_A_CONSULTAR:
    metars_mes = [m for m in todos_los_metars
                  if f'SKBO {str(año)[2:]}' in m or True]  # todos los del loop

    # Reagrupar por día para este mes
    reportes_por_dia = {}
    metars_filtrados = descargar_metars.__code__  # ya los tenemos en todos_los_metars

# Usamos directamente todos_los_metars ya descargados
reportes_por_dia_global = {}
for metar in todos_los_metars:
    match = re.search(r'SKBO (\d{2})(\d{2})(\d{2})Z', metar)
    if match:
        dia = int(match.group(1))
        # Identificar mes/año aproximado por posición en la lista
        clave = match.group(1)  # solo el día, suficiente para agrupar
        if clave not in reportes_por_dia_global:
            reportes_por_dia_global[clave] = []
        reportes_por_dia_global[clave].append(metar)

print('Días sin ninguna señal convectiva (buenos candidatos para día de control):')
print('=' * 60)
conteo = 0
for metar in todos_los_metars:
    tiene_conveccion = any(c in metar for c in ['CB', 'TCU', 'TSRA', 'TS', '+RA'])
    tiene_lluvia     = any(c in metar for c in ['RA', 'DZ', 'SH'])
    if not tiene_conveccion and not tiene_lluvia:
        match = re.search(r'SKBO (\d{6})Z', metar)
        if match and match.group(1)[2:4] == '00':  # solo el reporte de medianoche
            print(metar[:80])
            conteo += 1
            if conteo >= 10:
                print('... (mostrando primeros 10)')
                break
print('=' * 60)